# Episode 2: The shape of the work

## Why can one server coordinate 5,000 waiting tasks, yet 150 CPU-heavy tasks can stall it?

This notebook evaluates three claims:

1. **Waiting work:** threads and asyncio can overlap waiting, but a bounded thread pool processes excess work in waves while an event loop can park thousands of tasks.
2. **Pure-Python CPU work:** threads and asyncio do not create useful CPU parallelism on an ordinary GIL-enabled CPython build.
3. **Process workers:** separate processes make multi-core parallelism possible, but actual speedup depends on usable CPU capacity and job size.

The notebook uses only the Python standard library. Exact timings vary by machine. Interpret the mechanism and the shape of the result, not a memorized ratio.


## Hypotheses

Expected outcomes:

- At equal concurrency, threads and asyncio should both approach the 50 ms waiting floor; any difference mainly reflects runtime overhead.
- With 5,000 waiting jobs and 256 thread workers, the bounded pool should require about 20 waves. Asyncio can keep all 5,000 timers outstanding.
- Threads and asyncio should not meaningfully outperform serial execution for uninterrupted pure-Python CPU work on GIL-enabled CPython.
- A process pool may accelerate CPU work when enough independently usable CPU cores are available and each job is large enough to justify process overhead.


In [1]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from pathlib import Path
from statistics import median
from time import perf_counter, sleep
import asyncio
import importlib
import math
import multiprocessing
import os
import platform
import sys
import tempfile


print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Logical CPUs visible to Python: {os.cpu_count()}")

gil_probe = getattr(sys, "_is_gil_enabled", None)
if gil_probe:
    gil_status = gil_probe()
elif platform.python_implementation() == "CPython" and sys.version_info < (3, 13):
    gil_status = True
else:
    gil_status = "unknown on this Python version/build"

print(f"GIL enabled: {gil_status}")

if hasattr(os, "sched_getaffinity"):
    print(f"CPUs allowed by affinity: {len(os.sched_getaffinity(0))} {os.sched_getaffinity(0)}")

cpu_max = Path("/sys/fs/cgroup/cpu.max")
if cpu_max.exists():
    quota, period = cpu_max.read_text(encoding="utf-8").strip().split()
    print(f"cgroup cpu.max: {quota} {period}")
    if quota == "max":
        print("No explicit cgroup CPU quota")
    else:
        print(f"Effective cgroup CPU quota: {int(quota) / int(period):.2f}")


Python: 3.11.18
Platform: macOS-27.2-arm64-arm-64bit
Logical CPUs visible to Python: 10
GIL enabled: True


## Configuration

The thread model is deliberately bounded at 256 workers. It does **not** create 5,000 native threads. Therefore the scale experiment is a capacity comparison, not an equal-concurrency scheduler benchmark.


In [2]:
WAIT_TASKS = 5_000
WAIT_SECONDS = 0.050
THREAD_WAIT_WORKERS = 256
FAIR_WAIT_TASKS = THREAD_WAIT_WORKERS

CPU_TASKS = 150
PRIME_LIMIT = 16_000
CPU_THREAD_WORKERS = min(32, (os.cpu_count() or 1) + 4)

VISIBLE_CPUS = os.cpu_count() or 1

# Controlled experiment: use every visible CPU, capped at four.
# A two-CPU Colab runtime therefore creates two process workers.
PROCESS_WORKERS = max(1, min(4, VISIBLE_CPUS))

# Production services often start lower to preserve networking/OS headroom.
PRODUCTION_WORKER_START = max(1, min(4, VISIBLE_CPUS - 1))

REPEATS = 5

print({
    "waiting_tasks": WAIT_TASKS,
    "wait_seconds": WAIT_SECONDS,
    "bounded_thread_workers": THREAD_WAIT_WORKERS,
    "equal_concurrency_probe_tasks": FAIR_WAIT_TASKS,
    "cpu_tasks": CPU_TASKS,
    "prime_limit": PRIME_LIMIT,
    "cpu_thread_workers": CPU_THREAD_WORKERS,
    "experiment_process_workers": PROCESS_WORKERS,
    "production_worker_starting_point": PRODUCTION_WORKER_START,
    "repeats": REPEATS,
})


{'waiting_tasks': 5000, 'wait_seconds': 0.05, 'bounded_thread_workers': 256, 'equal_concurrency_probe_tasks': 256, 'cpu_tasks': 150, 'prime_limit': 16000, 'cpu_thread_workers': 14, 'experiment_process_workers': 4, 'production_worker_starting_point': 4, 'repeats': 5}


## Measurement helpers

Every benchmark receives one untimed warm-up when practical, runs five measured trials, and reports the median. A difference smaller than 5% is classified as effectively equal rather than treated as meaningful.


In [3]:
def timed(label, fn, repeats=REPEATS):
    samples = []
    last_value = None
    for _ in range(repeats):
        start = perf_counter()
        last_value = fn()
        samples.append(perf_counter() - start)
    return {
        "label": label,
        "seconds_median": median(samples),
        "seconds_all": samples,
        "value": last_value,
    }


async def timed_async(label, factory, repeats=REPEATS):
    samples = []
    last_value = None
    for _ in range(repeats):
        start = perf_counter()
        last_value = await factory()
        samples.append(perf_counter() - start)
    return {
        "label": label,
        "seconds_median": median(samples),
        "seconds_all": samples,
        "value": last_value,
    }


def count_primes(limit):
    count = 0
    for candidate in range(2, limit):
        prime = True
        divisor = 2
        while divisor * divisor <= candidate:
            if candidate % divisor == 0:
                prime = False
                break
            divisor += 1
        count += int(prime)
    return count


def print_results(title, records):
    print(f"\n{title}")
    print("-" * len(title))
    width = max(len(record["label"]) for record in records)
    for record in records:
        samples = ", ".join(f"{x:.4f}" for x in record["seconds_all"])
        print(f"{record['label']:<{width}}  median={record['seconds_median']:.4f}s  samples=[{samples}]")


def describe_change(baseline_record, candidate_record):
    ratio = baseline_record["seconds_median"] / candidate_record["seconds_median"]
    if abs(ratio - 1.0) < 0.05:
        return f"{candidate_record['label']}: effectively equal to baseline ({ratio:.2f}x)"
    if ratio > 1:
        return f"{candidate_record['label']}: {ratio:.2f}x faster than baseline"
    return f"{candidate_record['label']}: {1 / ratio:.2f}x slower than baseline"


benchmark_results = {}


## First: why `await count_primes(...)` does not work

`await` accepts an awaitable object. `count_primes()` performs the whole calculation and returns an integer. Python completes the calculation first, then tries to await that integer.


In [4]:
async def demonstrate_invalid_cpu_await():
    try:
        return await count_primes(2_000)
    except TypeError as error:
        return f"await failed as expected: {error}"


print(await demonstrate_invalid_cpu_await())


await failed as expected: object int can't be used in 'await' expression


## Experiment 1A: equal-concurrency waiting probe

Both models handle 256 simultaneous 50 ms waits. This is the fairer overhead comparison.

`sleep()` is a synthetic stand-in for waiting. It isolates scheduling behavior; it does not model sockets, payloads, database pools, or network backpressure.


In [5]:
def threaded_wait(pool, task_count):
    list(pool.map(lambda _: sleep(WAIT_SECONDS), range(task_count)))
    return task_count


async def async_wait(task_count):
    await asyncio.gather(*(asyncio.sleep(WAIT_SECONDS) for _ in range(task_count)))
    return task_count


wait_pool = ThreadPoolExecutor(max_workers=THREAD_WAIT_WORKERS)

try:
    # Warm all native workers before timing.
    list(wait_pool.map(lambda _: sleep(0), range(THREAD_WAIT_WORKERS)))
    await async_wait(8)

    fair_thread_record = timed(
        f"threads ({FAIR_WAIT_TASKS} tasks / {THREAD_WAIT_WORKERS} workers)",
        lambda: threaded_wait(wait_pool, FAIR_WAIT_TASKS),
    )
    fair_async_record = await timed_async(
        f"asyncio ({FAIR_WAIT_TASKS} tasks / one loop thread)",
        lambda: async_wait(FAIR_WAIT_TASKS),
    )
finally:
    wait_pool.shutdown(wait=True)

benchmark_results["waiting_equal"] = [fair_thread_record, fair_async_record]
print_results("Equal-concurrency waiting probe", benchmark_results["waiting_equal"])
print(describe_change(fair_thread_record, fair_async_record))



Equal-concurrency waiting probe
-------------------------------
threads (256 tasks / 256 workers)      median=0.0590s  samples=[0.0656, 0.0578, 0.0590, 0.0598, 0.0574]
asyncio (256 tasks / one loop thread)  median=0.0543s  samples=[0.0543, 0.0544, 0.0534, 0.0539, 0.0556]
asyncio (256 tasks / one loop thread): 1.09x faster than baseline


## Experiment 1B: 5,000-task capacity comparison

Now the models intentionally have different capacity:

- The bounded thread pool has 256 native workers, so 5,000 jobs require about 20 waves.
- The event loop can park all 5,000 timer tasks at once on one thread.

This comparison demonstrates scalable waiting capacity. It is **not** proof that asyncio has a universal N-times-faster scheduler.


In [6]:
thread_waves = math.ceil(WAIT_TASKS / THREAD_WAIT_WORKERS)
thread_floor = thread_waves * WAIT_SECONDS
async_floor = WAIT_SECONDS

print(f"Bounded thread-pool waves: {thread_waves}")
print(f"Approximate thread wait floor: {thread_floor:.2f}s")
print(f"Approximate asyncio wait floor: {async_floor:.2f}s plus task overhead")

wait_pool = ThreadPoolExecutor(max_workers=THREAD_WAIT_WORKERS)

try:
    list(wait_pool.map(lambda _: sleep(0), range(THREAD_WAIT_WORKERS)))
    await async_wait(8)

    scale_thread_record = timed(
        f"bounded thread pool ({WAIT_TASKS} jobs / {THREAD_WAIT_WORKERS} workers)",
        lambda: threaded_wait(wait_pool, WAIT_TASKS),
    )
    scale_async_record = await timed_async(
        f"asyncio ({WAIT_TASKS} outstanding tasks / one loop thread)",
        lambda: async_wait(WAIT_TASKS),
    )
finally:
    wait_pool.shutdown(wait=True)

benchmark_results["waiting_scale"] = [scale_thread_record, scale_async_record]
print_results("5,000-task waiting-capacity comparison", benchmark_results["waiting_scale"])
print("Interpretation: the ratio includes the thread pool's intentional 256-worker capacity limit.")


Bounded thread-pool waves: 20
Approximate thread wait floor: 1.00s
Approximate asyncio wait floor: 0.05s plus task overhead

5,000-task waiting-capacity comparison
--------------------------------------
bounded thread pool (5000 jobs / 256 workers)       median=1.0954s  samples=[1.0738, 1.1041, 1.0903, 1.0954, 1.1113]
asyncio (5000 outstanding tasks / one loop thread)  median=0.0885s  samples=[0.0983, 0.0773, 0.0885, 0.0886, 0.0775]
Interpretation: the ratio includes the thread pool's intentional 256-worker capacity limit.


## Experiment 2: pure-Python CPU work

Every task counts primes and contains no I/O and no suspension point.

- **Serial:** honest baseline.
- **Threads:** share one GIL-enabled interpreter, so they do not provide useful multi-core Python execution.
- **Asyncio:** `gather()` creates tasks, but each task runs an uninterrupted CPU function. It does not create parallelism.

This measures throughput, not event-loop responsiveness. A separate event-loop-lag experiment would be required to quantify how late timers and socket callbacks become.


In [7]:
def serial_cpu_once():
    return [count_primes(PRIME_LIMIT) for _ in range(CPU_TASKS)]


def threaded_cpu_once():
    with ThreadPoolExecutor(max_workers=CPU_THREAD_WORKERS) as pool:
        return list(pool.map(count_primes, [PRIME_LIMIT] * CPU_TASKS))


async def async_cpu_once():
    async def one_task():
        # Deliberately no await: this blocks the event-loop thread until done.
        return count_primes(PRIME_LIMIT)

    return await asyncio.gather(*(one_task() for _ in range(CPU_TASKS)))


# Untimed warm-ups.
count_primes(PRIME_LIMIT)
with ThreadPoolExecutor(max_workers=2) as warm_pool:
    list(warm_pool.map(count_primes, [2_000, 2_000]))
await async_cpu_once()

serial_record = timed("serial", serial_cpu_once)
thread_record = timed(
    f"threads ({CPU_THREAD_WORKERS} workers)",
    threaded_cpu_once,
)
async_record = await timed_async(
    "asyncio gather (CPU tasks do not yield)",
    async_cpu_once,
)

cpu_records = [serial_record, thread_record, async_record]
benchmark_results["cpu"] = cpu_records

assert len({tuple(record["value"]) for record in cpu_records}) == 1
print_results("Pure-Python CPU workload", cpu_records)
print(describe_change(serial_record, thread_record))
print(describe_change(serial_record, async_record))



Pure-Python CPU workload
------------------------
serial                                   median=1.4528s  samples=[1.4528, 1.4585, 1.4588, 1.4498, 1.4525]
threads (14 workers)                     median=1.4926s  samples=[1.4926, 1.4873, 1.4936, 1.4878, 1.5892]
asyncio gather (CPU tasks do not yield)  median=1.5268s  samples=[1.5357, 1.5051, 1.4750, 1.5268, 1.5531]
threads (14 workers): effectively equal to baseline (0.97x)
asyncio gather (CPU tasks do not yield): effectively equal to baseline (0.95x)


## Experiment 3: persistent process workers

Processes have separate interpreters and GILs, so they can execute pure-Python work on independently usable cores.

The pool is created once, warmed, and reused. Process startup is therefore not charged to every measured trial. The direct pool timing isolates CPU throughput. After that, the notebook demonstrates the correct `await` integration separately.


In [8]:
worker_dir = Path(tempfile.gettempdir()) / "episode_2_workers"
worker_dir.mkdir(parents=True, exist_ok=True)
worker_path = worker_dir / "episode_2_worker.py"
worker_path.write_text(
    """def count_primes(limit):
    count = 0
    for candidate in range(2, limit):
        prime = True
        divisor = 2
        while divisor * divisor <= candidate:
            if candidate % divisor == 0:
                prime = False
                break
            divisor += 1
        count += int(prime)
    return count
""",
    encoding="utf-8",
)

if str(worker_dir) not in sys.path:
    sys.path.insert(0, str(worker_dir))

importlib.invalidate_caches()
episode_2_worker = importlib.import_module("episode_2_worker")


def direct_process_cpu_once(pool):
    return list(
        pool.map(
            episode_2_worker.count_primes,
            [PRIME_LIMIT] * CPU_TASKS,
            chunksize=1,
        )
    )


async def awaited_process_cpu_once(pool):
    loop = asyncio.get_running_loop()
    futures = [
        loop.run_in_executor(
            pool,
            episode_2_worker.count_primes,
            PRIME_LIMIT,
        )
        for _ in range(CPU_TASKS)
    ]
    return await asyncio.gather(*futures)


multiprocessing.freeze_support()
process_pool = ProcessPoolExecutor(max_workers=PROCESS_WORKERS)

try:
    # Start every worker before measuring the reusable pool.
    warmup = [
        process_pool.submit(episode_2_worker.count_primes, 2_000)
        for _ in range(PROCESS_WORKERS)
    ]
    [future.result() for future in warmup]

    process_record = timed(
        f"persistent processes ({PROCESS_WORKERS} workers)",
        lambda: direct_process_cpu_once(process_pool),
    )

    # Correct await demonstration: futures are awaitable; count_primes() is not.
    awaited_process_result = await awaited_process_cpu_once(process_pool)
finally:
    process_pool.shutdown(wait=True)

benchmark_results["process"] = [process_record]
assert tuple(process_record["value"]) == tuple(serial_record["value"])
assert tuple(awaited_process_result) == tuple(serial_record["value"])

print_results("Process-pool CPU workload", [process_record])
print(describe_change(serial_record, process_record))
print("Correct await confirmed: the event loop awaited process-pool futures successfully.")

if serial_record["seconds_median"] / process_record["seconds_median"] <= 1.05:
    print("No process speedup on this runtime: visible workers did not provide additional effective throughput, or overhead dominated.")



Process-pool CPU workload
-------------------------
persistent processes (4 workers)  median=0.5111s  samples=[0.5407, 0.5204, 0.5111, 0.4831, 0.5024]
persistent processes (4 workers): 2.84x faster than baseline
Correct await confirmed: the event loop awaited process-pool futures successfully.


## What `await` means in the correct architecture

Invalid:

```python
await count_primes(16_000)  # calculation completes, then Python tries to await an integer
```

Correct:

```python
future = loop.run_in_executor(process_pool, count_primes, 16_000)
result = await future
```

The process performs the computation. The event loop awaits a future representing completion, so it does not execute the CPU function itself.


## Results summary

The next cell prints only the measured medians. Interpret the categories separately:

- `waiting_equal`: equal-concurrency overhead probe;
- `waiting_scale`: intentionally different capacity limits;
- `cpu`: no meaningful thread/async parallelism is expected;
- `process`: speedup depends on independently usable CPU capacity.


In [9]:
summary_rows = []
for shape, records in benchmark_results.items():
    for record in records:
        summary_rows.append((shape, record["label"], record["seconds_median"]))

shape_width = max(len(row[0]) for row in summary_rows)
model_width = max(len(row[1]) for row in summary_rows)

print(f"{'experiment':<{shape_width}}  {'model':<{model_width}}  median seconds")
print(f"{'-' * shape_width}  {'-' * model_width}  --------------")
for shape, model, seconds in summary_rows:
    print(f"{shape:<{shape_width}}  {model:<{model_width}}  {seconds:14.4f}")


experiment     model                                               median seconds
-------------  --------------------------------------------------  --------------
waiting_equal  threads (256 tasks / 256 workers)                           0.0590
waiting_equal  asyncio (256 tasks / one loop thread)                       0.0543
waiting_scale  bounded thread pool (5000 jobs / 256 workers)               1.0954
waiting_scale  asyncio (5000 outstanding tasks / one loop thread)          0.0885
cpu            serial                                                      1.4528
cpu            threads (14 workers)                                        1.4926
cpu            asyncio gather (CPU tasks do not yield)                     1.5268
process        persistent processes (4 workers)                            0.5111


## Conclusions

The equal-concurrency probe compares overhead. The 5,000-task experiment compares capacity: a bounded 256-thread pool works in waves, while one event loop parks all 5,000 waiting tasks. For uninterrupted pure-Python CPU work, threads and asyncio remain on the serial line. A persistent process pool makes multi-core execution possible when the runtime supplies independently usable cores. The event loop holds the waiters; worker processes spend the usable cores.

### Production caveats

- Real image libraries may release the GIL or start native threads; benchmark the actual library.
- `os.cpu_count()` reports visible logical CPUs, not guaranteed physical throughput.
- Bound worker counts, queues, memory, retries, and upload sizes.
- An unbounded queue hides overload; it does not solve it.
- Re-measure after every architectural change because the bottleneck can move.
